# Metrics

Метрики классификации и регрессии на NumPy (из `metrics.py`).

In [1]:
import numpy as np

## Классификация

In [2]:
def _conf(y, p):
    y, p = np.asarray(y, dtype=bool), np.asarray(p, dtype=bool)
    tp = int(np.sum(y & p))
    tn = int(np.sum(~y & ~p))
    fp = int(np.sum(~y & p))
    fn = int(np.sum(y & ~p))
    return tp, tn, fp, fn


def accuracy(y, p):
    y, p = np.asarray(y), np.asarray(p)
    return float(np.mean(y == p))


def precision(y, p):
    tp, _, fp, _ = _conf(y, p)
    return tp / (tp + fp) if tp + fp else 0.0


def recall(y, p):
    tp, _, _, fn = _conf(y, p)
    return tp / (tp + fn) if tp + fn else 0.0


def f1(y, p):
    tp, _, fp, fn = _conf(y, p)
    denom = 2 * tp + fp + fn
    return 2 * tp / denom if denom else 0.0


def roc_auc(y, s):
    y, s = np.asarray(y), np.asarray(s, dtype=float)
    n1 = int(y.sum())
    n0 = y.size - n1
    ranks = np.empty(s.size)
    ranks[np.argsort(s, kind="mergesort")] = np.arange(1, s.size + 1)
    _, inv, cnt = np.unique(s, return_inverse=True, return_counts=True)
    ranks = (np.bincount(inv, weights=ranks) / cnt)[inv]
    return float((ranks[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


def log_loss(y, p, eps=1e-15):
    y, p = np.asarray(y, dtype=float), np.asarray(p, dtype=float)
    p = np.clip(p, eps, 1 - eps)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))

## Регрессия

In [3]:
def mse(y, p):
    y, p = np.asarray(y, dtype=float), np.asarray(p, dtype=float)
    return float(np.mean((y - p) ** 2))


def mae(y, p):
    y, p = np.asarray(y, dtype=float), np.asarray(p, dtype=float)
    return float(np.mean(np.abs(y - p)))


def mape(y, p):
    y, p = np.asarray(y, dtype=float), np.asarray(p, dtype=float)
    denom = np.maximum(np.abs(y), np.finfo(float).eps)
    return float(np.mean(np.abs(y - p) / denom) * 100)


def r2(y, p):
    y, p = np.asarray(y, dtype=float), np.asarray(p, dtype=float)
    ss_res = np.sum((y - p) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    if ss_tot == 0:
        return 1.0 if ss_res == 0 else 0.0
    return float(1 - ss_res / ss_tot)